In [11]:
import os
import warnings

# Suppress HuggingFace warnings and tqdm output
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")

In [36]:
# %pip install langchain-chroma
# %pip install pyprojroot
# %pip install langchain_huggingface
# %pip install sentence-transformers

In [2]:
from langchain_chroma import Chroma
import os
from pyprojroot import here
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pprint import pprint
load_dotenv()

True

**Load environment variables and configs**

In [12]:
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
VECTORDB_DIR = "data/airline_policy_vectordb"
K=2

**Load the vectorDB**

In [13]:
load_dotenv()
vectordb = Chroma(
    collection_name="rag-chroma",
    persist_directory=str(here(VECTORDB_DIR)),
    embedding_function=HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
)
print("Number of vectors in vectordb:",
      vectordb._collection.count(), "\n\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7078.12it/s]


Number of vectors in vectordb: 22 




**Sample Query**

In [5]:
message = "What is the cancelation rule for a flight ticket at swiss airline policy?"

**Perform the vector Search**

In [6]:
docs = vectordb.similarity_search(message, k=K)

In [7]:
docs

[Document(id='2dbd99e6-b5e0-4ea3-944b-9151388008c0', metadata={'creationdate': '2024-09-08T21:34:59+00:00', 'creator': 'PDFCreator Online (www.pdfforge.org/online)', 'title': 'Merged with PDFCreator Online', 'source': '/Users/shefalisingh/Desktop/SQL_agent/data/unstructred_data/airline_policy/swiss_faq.pdf', 'total_pages': 12, 'producer': 'PDFCreator Online (www.pdfforge.org/online)', 'page_label': '9', 'page': 8, 'moddate': '2024-09-08T21:34:59+00:00'}, page_content="How to Cancel a Swiss Air Flight: 877-\n5O7-7341 Step-by-Step Guide\nSwiss Air is a premium airline based in Switzerland that of fers a range of domestic and international flights to\npassengers. However , sometimes situations arise where passengers may need to cancel their flights. In such cases, it is\nimportant to understand the Swiss Air Cancellation Policy to avoid any confusion or additional charges.\nSwiss International Airlines Cancellation Policy In this article, we will provide you with everything you need to kn

**Prepare the prompt for the Groq model**

In [8]:
question = "# User new question:\n" + message
retrieved_content = ""
for doc in docs:
    retrieved_content += f"{doc.page_content}\n\n"
prompt = f"# Content:\n{retrieved_content}\n\n{question}"

Prepared prompt

In [9]:
pprint(prompt)

('# Content:\n'
 'How to Cancel a Swiss Air Flight: 877-\n'
 '5O7-7341 Step-by-Step Guide\n'
 'Swiss Air is a premium airline based in Switzerland that of fers a range of '
 'domestic and international flights to\n'
 'passengers. However , sometimes situations arise where passengers may need '
 'to cancel their flights. In such cases, it is\n'
 'important to understand the Swiss Air Cancellation Policy to avoid any '
 'confusion or additional charges.\n'
 'Swiss International Airlines Cancellation Policy In this article, we will '
 'provide you with everything you need to know about\n'
 'the Swiss Air Cancellation Policy , including how to cancel a Swiss Air '
 'flight, the fees associated with cancelling a flight,\n'
 'and the refund policy .\n'
 "If you have booked a flight with Swiss Airlines but need to cancel it, it's "
 'important to understand their cancellation policy\n'
 'to avoid any unnecessary fees or charges. Swiss Airlines of fers dif ferent '
 'fare types, each with thei

**Pass the prompt to the Groq model and get the response**

In [14]:
load_dotenv()
llm = ChatGroq(
    model_name="qwen/qwen3.8-27b",
    groq_api_key=os.environ['GROQ_API_KEY']
)

response = llm.invoke([
    ("system", "You will receive a user's query and possible content where the answer might be. If the answer is found, provide it, if not, state that the answer does not exist."),
    ("user", prompt)
])

Printing the response

In [15]:
print(response.content)

Based on the content provided, the cancellation rules for Swiss Air tickets depend primarily on the **fare type** and the **timing** of the cancellation:

1.  **Flexible Fare Types (Flex and Business Flex):**
    *   You can cancel up to **24 hours before departure** without any penalty.

2.  **Other Fare Types (e.g., Non-flexible Economy):**
    *   **Within 24 hours of booking:** Implied 24-hour cancellation policy (standard airline practice, though the text cuts off, it mentions "Swiss Airlines 24 Hour Cancellation Policy").
    *   **Outside the 24-hour window:** Cancellation fees apply. The fee amount depends on the specific fare type and how close the cancellation is to the departure date. Generally, the closer to the departure date, the higher the fee.

3.  **If Swiss Airlines Cancels the Flight:**
    *   You may be entitled to a **full refund** or rebooking on another flight.
    *   **Exception:** If the cancellation is due to **extraordinary circumstances** (e.g., bad weathe

**RAG Tool design using LangChain**

In [16]:
from langchain_core.tools import tool

@tool
def lookup_swiss_airline_policy(query: str)->str:
    """Search within the Swiss Airline's company policies to check whether certain options are permitted. Input should be a search query."""
    vectordb = Chroma(
    collection_name="rag-chroma",
    persist_directory=str(here(VECTORDB_DIR)),
    embedding_function=HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
    )
    docs = vectordb.similarity_search(query, k=K)
    return "\n\n".join([doc.page_content for doc in docs])


In [17]:
print(lookup_swiss_airline_policy.name)
print(lookup_swiss_airline_policy.args)
print(lookup_swiss_airline_policy.description)

lookup_swiss_airline_policy
{'query': {'title': 'Query', 'type': 'string'}}
Search within the Swiss Airline's company policies to check whether certain options are permitted. Input should be a search query.


In [18]:
raw_context = lookup_swiss_airline_policy.invoke("can I cancel my ticket?")

prompt = f"""Use the following context to answer the user question cleanly and clearly.
Context:
{raw_context}
User Question: Can I cancel my ticket?
"""

response = llm.invoke(prompt)
print(response.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7723.49it/s]


Yes, you can cancel your Swiss Airlines ticket, but the terms depend on when you cancel:

*   **Within 24 hours of booking:** You can cancel **without penalty** and receive a **full refund** of the ticket price. This policy applies to all fare types, including non-refundable tickets, provided the flight was booked directly through Swiss Airlines.
*   **After 24 hours:** You may be subject to cancellation fees or penalties, and eligibility for a refund depends on your specific ticket type and terms.

**How to cancel within 24 hours:**
1.  Go to the Swiss Airlines website and click on **"Manage your bookings."**
2.  Enter your booking reference number and last name.
3.  Select the flight you wish to cancel and click **"Cancel flight."**
4.  Confirm the cancellation to receive your full refund.

**Important Notes:**
*   If you booked through a **travel agent or third-party website**, you must contact them directly, as their cancellation policies may differ from Swiss Airlines’.
*   In cas